# SEOBNRv5HMROM frequency-domain waveform scan over one FTI parameter

This notebook generates frequency-domain `SEOBNRv5HMROM` waveforms while varying one FTI phase-deformation parameter over a chosen range.

Default interpretation used here:

```python
fti_parameter = "fti_dchi2v"
```

So this scans the coefficient usually called `dchi2`, the FTI correction associated with the `v^2` PN-like phase term.

If by "2nd FTI parameter" you meant a different parameter, change only this line:

```python
fti_parameter = "fti_dchi2v"
```

Available aliases are listed below, for example:

```python
"fti_dchiMinus2v", "fti_dchi0v", "fti_dchi1v", "fti_dchi2v",
"fti_dchi3v", "fti_dchi4v", "fti_dchi5v", "fti_dchi5lv",
"fti_dchi6v", "fti_dchi6lv", "fti_dchi7v"
```

The notebook saves:

- raw complex FD waveforms in `.npz`
- a summary table in `.csv`
- comparison plots for amplitude, phase difference, and relative amplitude difference

In [1]:
# If you run this in Jupyter:
%matplotlib inline

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.linestyle": ":",
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 10,
})

# Output directory
outdir = Path("seobnrv5hmrom_fd_fti_scan_outputs")
outdir.mkdir(parents=True, exist_ok=True)
print("Saving outputs to:", outdir.resolve())

Saving outputs to: /pbs/home/a/amahdi/Simulating deviations/seobnrv5hmrom_fd_fti_scan_outputs


## 1. Import LALSuite

Run this notebook in the same environment where your SEOBNRv5HMROM / lisabeta jobs run.

For example, on the cluster this may be something like:

```bash
conda activate lisabeta-py310
source /path/to/lalsuiterc
jupyter notebook
```

In [2]:
try:
    import lal
    import lalsimulation as lalsim
except Exception as exc:
    raise ImportError(
        "Could not import lal/lalsimulation. Run this notebook in your "
        "LALSuite/lisabeta environment, then re-run this cell."
    ) from exc

print("lal imported")
print("lalsimulation imported")
print("SEOBNRv5HMROM approximant ID:", lalsim.GetApproximantFromString("SEOBNRv5HM_ROM"))

lal imported
lalsimulation imported
SEOBNRv5HMROM approximant ID: 114


## 2. Physical parameters

These are your usual `M = 10^6 M_\odot`, `q = 4`, aligned-spin example values.

For LAL, masses must be passed in SI units, distance in meters, and angles in radians.

In [3]:
# -------------------------------------------------------------------------
# Source parameters
# -------------------------------------------------------------------------
source = {
    "M": 1.0e6,                         # total detector-frame mass in solar masses
    "q": 4.0,                           # m1 / m2, with m1 >= m2
    "chi1": 0.5,                        # aligned spin of primary
    "chi2": 0.5,                        # aligned spin of secondary
    "dist_mpc": 6791.810623950696,      # luminosity distance in Mpc
    "inc": np.pi / 3,                   # inclination
    "phi_ref": 0.7,                     # reference orbital phase
}

# -------------------------------------------------------------------------
# Frequency-domain controls
# -------------------------------------------------------------------------
f_min = 1.0e-5
f_max = 2.0e-1

# Choose delta_f small enough for a smooth plot, but not too heavy.
# For a 3-day observation, 1/T_obs = 1/(3*86400) = 3.86e-6 Hz.
# You can use that value if you want consistency with your earlier runs.
T_obs = 3 * 86400.0
delta_f = 1.0 / T_obs

# Reference frequency. For ROMs it is often safest to use f_min.
f_ref = f_min

# Optional: restrict modes used by the approximant.
# Set modes = None to let the model use its default modes.
modes = [(2, 2), (2, 1), (3, 3), (4, 4), (4, 3), (5, 5)]

print("delta_f =", delta_f)
print("Number of FD bins roughly =", int(np.ceil(f_max / delta_f)))

delta_f = 3.858024691358025e-06
Number of FD bins roughly = 51840


## 3. Choose the FTI parameter and scan range

Default:

```python
fti_parameter = "fti_dchi2v"
```

This is the `dchi2` phase deformation. You can replace it by any FTI parameter alias in the dictionary below.

In [4]:
# -------------------------------------------------------------------------
# FTI parameter to vary
# -------------------------------------------------------------------------
fti_parameter = "fti_dchi2v"

# Scan values.
# You can make this narrower if the phase variation is too large.
fti_values = np.array([
    -0.50, -0.20, -0.10, -0.05,
     0.00,
     0.05,  0.10,  0.20,  0.50
], dtype=float)

print("FTI parameter:", fti_parameter)
print("FTI scan values:", fti_values)

FTI parameter: fti_dchi2v
FTI scan values: [-0.5  -0.2  -0.1  -0.05  0.    0.05  0.1   0.2   0.5 ]


## 4. Helper functions

The most important helper is `insert_fti_parameter`, which inserts the selected non-GR phase coefficient into the LAL dictionary.

The notebook tries the standard LALSimulation insertion functions, such as:

```python
lalsim.SimInspiralWaveformParamsInsertNonGRDChi2(...)
```

for `dchi2`.

In [5]:
# -------------------------------------------------------------------------
# Aliases for FTI / parameterized-test phase deformation names
# -------------------------------------------------------------------------
FTI_ALIASES = {
    # -2 PN / dipole-like term
    "dchiminus2": "dchiMinus2",
    "dchi_minus2": "dchiMinus2",
    "dchi-2": "dchiMinus2",
    "dchiMinus2": "dchiMinus2",
    "fti_dchiminus2v": "dchiMinus2",
    "fti_dchiMinus2v": "dchiMinus2",
    "fti_dchiMinus2": "dchiMinus2",

    # regular PN-like terms
    "dchi0": "dchi0",
    "dchi0v": "dchi0",
    "fti_dchi0v": "dchi0",

    "dchi1": "dchi1",
    "dchi1v": "dchi1",
    "fti_dchi1v": "dchi1",

    "dchi2": "dchi2",
    "dchi2v": "dchi2",
    "fti_dchi2v": "dchi2",

    "dchi3": "dchi3",
    "dchi3v": "dchi3",
    "fti_dchi3v": "dchi3",

    "dchi4": "dchi4",
    "dchi4v": "dchi4",
    "fti_dchi4v": "dchi4",

    "dchi5": "dchi5",
    "dchi5v": "dchi5",
    "fti_dchi5v": "dchi5",

    "dchi5l": "dchi5L",
    "dchi5L": "dchi5L",
    "dchi5lv": "dchi5L",
    "dchi5Lv": "dchi5L",
    "fti_dchi5lv": "dchi5L",
    "fti_dchi5Lv": "dchi5L",

    "dchi6": "dchi6",
    "dchi6v": "dchi6",
    "fti_dchi6v": "dchi6",

    "dchi6l": "dchi6L",
    "dchi6L": "dchi6L",
    "dchi6lv": "dchi6L",
    "dchi6Lv": "dchi6L",
    "fti_dchi6lv": "dchi6L",
    "fti_dchi6Lv": "dchi6L",

    "dchi7": "dchi7",
    "dchi7v": "dchi7",
    "fti_dchi7v": "dchi7",
}


def canonical_fti_name(name):
    # Return canonical FTI name used internally.
    if name in FTI_ALIASES:
        return FTI_ALIASES[name]

    compact = str(name).replace("_", "").replace("-", "").replace(" ", "")
    if compact in FTI_ALIASES:
        return FTI_ALIASES[compact]

    raise ValueError(
        f"Unknown FTI parameter {name!r}. Available aliases include:\n"
        + "\n".join(sorted(FTI_ALIASES.keys()))
    )


def fti_lal_suffix(canonical):
    # Map canonical dchi name to the suffix used in LAL function names.
    mapping = {
        "dchiMinus2": "Minus2",
        "dchi0": "0",
        "dchi1": "1",
        "dchi2": "2",
        "dchi3": "3",
        "dchi4": "4",
        "dchi5": "5",
        "dchi5L": "5L",
        "dchi6": "6",
        "dchi6L": "6L",
        "dchi7": "7",
    }
    return mapping[canonical]


def insert_fti_parameter(lal_dict, parameter_name, value):
    """Insert one FTI / non-GR phase deformation parameter into a LAL dict.

    Example:
        insert_fti_parameter(lal_dict, "fti_dchi2v", 0.1)

    This calls a LAL function like:
        SimInspiralWaveformParamsInsertNonGRDChi2(lal_dict, 0.1)
    """
    canonical = canonical_fti_name(parameter_name)
    suffix = fti_lal_suffix(canonical)

    candidates = [
        f"SimInspiralWaveformParamsInsertNonGRDChi{suffix}",
        f"SimInspiralWaveformParamsInsertNonGRDChi{suffix.upper()}",
    ]

    for func_name in candidates:
        func = getattr(lalsim, func_name, None)
        if func is not None:
            func(lal_dict, float(value))
            return canonical, func_name

    available = [x for x in dir(lalsim) if "InsertNonGRDChi" in x]
    raise AttributeError(
        "Could not find the needed LAL insertion function for "
        f"{parameter_name!r} -> {canonical!r}.\n"
        f"Tried: {candidates}\n\n"
        "Functions found in this lalsimulation build:\n"
        + "\n".join(available)
    )


def make_mode_array(modes):
    # Create a LAL mode array. Return None if modes is None.
    if modes is None:
        return None

    mode_array = lalsim.SimInspiralCreateModeArray()
    for ell, emm in modes:
        lalsim.SimInspiralModeArrayActivateMode(mode_array, int(ell), int(emm))
    return mode_array


def build_lal_dict(fti_parameter=None, fti_value=0.0, modes=None):
    # Create the LAL parameter dictionary for this waveform.
    lal_dict = lal.CreateDict()

    # Optional higher-mode restriction.
    if modes is not None:
        mode_array = make_mode_array(modes)
        lalsim.SimInspiralWaveformParamsInsertModeArray(lal_dict, mode_array)

    insertion_info = None
    if fti_parameter is not None and float(fti_value) != 0.0:
        canonical, func_name = insert_fti_parameter(lal_dict, fti_parameter, fti_value)
        insertion_info = {
            "parameter_input": fti_parameter,
            "parameter_canonical": canonical,
            "lal_function": func_name,
            "value": float(fti_value),
        }

    return lal_dict, insertion_info


def source_to_lal_units(source):
    # Convert the source dictionary to LAL/SI units.
    M = float(source["M"])
    q = float(source["q"])

    m1_msun = M * q / (1.0 + q)
    m2_msun = M / (1.0 + q)

    return {
        "m1_msun": m1_msun,
        "m2_msun": m2_msun,
        "m1_si": m1_msun * lal.MSUN_SI,
        "m2_si": m2_msun * lal.MSUN_SI,
        "distance_si": float(source["dist_mpc"]) * 1.0e6 * lal.PC_SI,
        "spin1z": float(source.get("chi1", 0.0)),
        "spin2z": float(source.get("chi2", 0.0)),
        "inclination": float(source.get("inc", 0.0)),
        "phi_ref": float(source.get("phi_ref", 0.0)),
    }


def fd_series_to_numpy(series):
    # Convert a LAL COMPLEX16FrequencySeries to f and complex numpy array.
    data = np.asarray(series.data.data, dtype=np.complex128)
    f0 = float(series.f0)
    df = float(series.deltaF)
    f = f0 + df * np.arange(len(data))
    return f, data


def generate_seobnrv5hmrom_fd(source, f_min, f_max, delta_f, f_ref,
                              fti_parameter=None, fti_value=0.0, modes=None):
    # Generate FD hp, hc from SEOBNRv5HMROM, optionally with one FTI parameter.
    src = source_to_lal_units(source)
    lal_dict, insertion_info = build_lal_dict(
        fti_parameter=fti_parameter,
        fti_value=fti_value,
        modes=modes,
    )

    approximant = lalsim.GetApproximantFromString("SEOBNRv5HM_ROM")

    hp, hc = lalsim.SimInspiralChooseFDWaveform(
        src["m1_si"],
        src["m2_si"],
        0.0, 0.0, src["spin1z"],
        0.0, 0.0, src["spin2z"],
        src["distance_si"],
        src["inclination"],
        src["phi_ref"],
        0.0,        # longAscNodes
        0.0,        # eccentricity
        0.0,        # meanPerAno
        float(delta_f),
        float(f_min),
        float(f_max),
        float(f_ref),
        lal_dict,
        approximant,
    )

    f_hp, hp_np = fd_series_to_numpy(hp)
    f_hc, hc_np = fd_series_to_numpy(hc)

    if not np.allclose(f_hp, f_hc):
        raise RuntimeError("hp and hc frequency grids are different.")

    # Keep only the requested range.
    mask = (f_hp >= f_min) & (f_hp <= f_max)
    return {
        "f": f_hp[mask],
        "hp": hp_np[mask],
        "hc": hc_np[mask],
        "insertion_info": insertion_info,
    }


def safe_name(x):
    return str(x).replace("-", "m").replace("+", "p").replace(".", "p")

## 5. Test one GR waveform

First generate the GR baseline, without any FTI deformation.

In [6]:
gr = generate_seobnrv5hmrom_fd(
    source=source,
    f_min=f_min,
    f_max=f_max,
    delta_f=delta_f,
    f_ref=f_ref,
    fti_parameter=None,
    fti_value=0.0,
    modes=modes,
)

f = gr["f"]
hp_gr = gr["hp"]
hc_gr = gr["hc"]

amp_gr = np.abs(hp_gr)

print("Generated GR SEOBNRv5HMROM waveform")
print("Frequency range:", f[0], "to", f[-1], "Hz")
print("Number of samples:", len(f))
print("max |hp|:", np.nanmax(np.abs(hp_gr)))
print("max |hc|:", np.nanmax(np.abs(hc_gr)))

Mode (2,1) is not available by the model SEOBNRv5HM_ROM.
In this function you can only select (l,-|m|) modes that are directly modeled in the ROM.
The (l,+|m|) mode will be automatically added to the waveform using symmetry arguments.
Not available mode chosen.
XLAL Error - XLALSimIMRSEOBNRv5HMROM (LALSimIMRSEOBNRv5HMROM.c:2681): Internal function call failed
XLAL Error - XLALSimInspiralChooseFDWaveform_legacy (LALSimInspiralGeneratorLegacy.c:2786): Internal function call failed


RuntimeError: Internal function call failed

In [ ]:
plt.figure(figsize=(9, 5))
mask = (f > 0) & (np.abs(hp_gr) > 0)
plt.loglog(f[mask], np.abs(hp_gr[mask]), label=r"GR $|\tilde h_+|$")
plt.xlabel("frequency [Hz]")
plt.ylabel(r"$|\tilde h_+(f)|$")
plt.title("SEOBNRv5HMROM GR frequency-domain waveform")
plt.legend()
plt.tight_layout()
plt.savefig(outdir / "seobnrv5hmrom_gr_hp_amplitude.png", dpi=250)
plt.show()

## 6. Generate the FTI scan

This loops over all selected values of the chosen FTI parameter.

In [ ]:
scan = {}

for value in fti_values:
    print(f"Generating {fti_parameter} = {value:+.6g}")
    wf = generate_seobnrv5hmrom_fd(
        source=source,
        f_min=f_min,
        f_max=f_max,
        delta_f=delta_f,
        f_ref=f_ref,
        fti_parameter=fti_parameter,
        fti_value=value,
        modes=modes,
    )
    scan[float(value)] = wf

print("Done.")
print("Example insertion info for a nonzero value:")
for val, wf in scan.items():
    if abs(val) > 0:
        print(val, wf["insertion_info"])
        break

## 7. Save raw complex waveforms

The `.npz` file contains the frequency grid, GR waveform, and every scanned FTI waveform.

In [ ]:
npz_dict = {
    "f": f,
    "hp_gr": hp_gr,
    "hc_gr": hc_gr,
    "fti_values": fti_values,
}

for val, wf in scan.items():
    tag = safe_name(f"{val:+.6g}")
    npz_dict[f"hp_{tag}"] = wf["hp"]
    npz_dict[f"hc_{tag}"] = wf["hc"]

npz_path = outdir / f"seobnrv5hmrom_fd_scan_{canonical_fti_name(fti_parameter)}.npz"
np.savez_compressed(npz_path, **npz_dict)

print("Saved:", npz_path)

## 8. Plot amplitude comparison

For an FTI phase deformation, the leading effect should mainly appear in the phase. The amplitude curves may therefore lie almost on top of each other.

In [ ]:
plt.figure(figsize=(10, 5.6))

amp_mask = (f > 0) & (np.abs(hp_gr) > 0)
plt.loglog(f[amp_mask], np.abs(hp_gr[amp_mask]), linewidth=2.5, label="GR")

for val, wf in scan.items():
    if np.isclose(val, 0.0):
        continue
    hp = wf["hp"]
    mask = (f > 0) & (np.abs(hp) > 0)
    plt.loglog(f[mask], np.abs(hp[mask]), alpha=0.8, label=rf"{fti_parameter}={val:g}")

plt.xlabel("frequency [Hz]")
plt.ylabel(r"$|\tilde h_+(f)|$")
plt.title(rf"SEOBNRv5HMROM amplitude scan: {fti_parameter}")
plt.legend(ncol=2)
plt.tight_layout()

amp_plot_path = outdir / f"amplitude_scan_{canonical_fti_name(fti_parameter)}.png"
plt.savefig(amp_plot_path, dpi=250)
plt.show()

print("Saved:", amp_plot_path)

## 9. Plot phase difference relative to GR

This is usually the most useful diagnostic for FTI parameters.

The plotted quantity is

```python
Delta phase = unwrap(angle(h_FTI / h_GR))
```

computed only where the GR amplitude is safely above zero.

In [ ]:
# Use a threshold so phase is not dominated by numerical noise at tiny amplitudes.
amp_threshold = 1e-8 * np.nanmax(np.abs(hp_gr))
phase_mask = (f >= f_min) & (f <= f_max) & (np.abs(hp_gr) > amp_threshold)

plt.figure(figsize=(10, 5.6))

for val, wf in scan.items():
    if np.isclose(val, 0.0):
        continue

    hp = wf["hp"]
    # Relative phase is more stable than subtracting two separately unwrapped phases.
    dphase = np.unwrap(np.angle(hp[phase_mask] / hp_gr[phase_mask]))
    plt.plot(f[phase_mask], dphase, label=rf"{fti_parameter}={val:g}")

plt.xscale("log")
plt.xlabel("frequency [Hz]")
plt.ylabel(r"$\Delta \Psi_+(f)$ relative to GR [rad]")
plt.title(rf"SEOBNRv5HMROM FTI phase scan: {fti_parameter}")
plt.legend(ncol=2)
plt.tight_layout()

phase_plot_path = outdir / f"phase_difference_scan_{canonical_fti_name(fti_parameter)}.png"
plt.savefig(phase_plot_path, dpi=250)
plt.show()

print("Saved:", phase_plot_path)

## 10. Relative amplitude difference

This checks whether the scan is mainly phase-like, as expected.

In [ ]:
plt.figure(figsize=(10, 5.6))

relamp_mask = (f >= f_min) & (f <= f_max) & (np.abs(hp_gr) > amp_threshold)

for val, wf in scan.items():
    if np.isclose(val, 0.0):
        continue

    hp = wf["hp"]
    rel_amp = (np.abs(hp[relamp_mask]) - np.abs(hp_gr[relamp_mask])) / np.abs(hp_gr[relamp_mask])
    plt.plot(f[relamp_mask], rel_amp, label=rf"{fti_parameter}={val:g}")

plt.xscale("log")
plt.xlabel("frequency [Hz]")
plt.ylabel(r"$(|\tilde h_{\rm FTI}| - |\tilde h_{\rm GR}|)/|\tilde h_{\rm GR}|$")
plt.title(rf"Relative amplitude difference: {fti_parameter}")
plt.legend(ncol=2)
plt.tight_layout()

relamp_plot_path = outdir / f"relative_amplitude_difference_scan_{canonical_fti_name(fti_parameter)}.png"
plt.savefig(relamp_plot_path, dpi=250)
plt.show()

print("Saved:", relamp_plot_path)

## 11. Real part comparison in frequency domain

This is less physically clean than the phase plot, but it is useful for quickly seeing that the complex waveform itself changes.

In [ ]:
plt.figure(figsize=(10, 5.6))

plot_mask = (f >= f_min) & (f <= f_max) & (np.abs(hp_gr) > amp_threshold)
plt.plot(f[plot_mask], np.real(hp_gr[plot_mask]), linewidth=2.5, label="GR")

for val, wf in scan.items():
    if np.isclose(val, 0.0):
        continue
    hp = wf["hp"]
    plt.plot(f[plot_mask], np.real(hp[plot_mask]), alpha=0.8, label=rf"{fti_parameter}={val:g}")

plt.xscale("log")
plt.xlabel("frequency [Hz]")
plt.ylabel(r"Re$(\tilde h_+)$")
plt.title(rf"Real part of FD waveform: {fti_parameter}")
plt.legend(ncol=2)
plt.tight_layout()

real_plot_path = outdir / f"real_part_scan_{canonical_fti_name(fti_parameter)}.png"
plt.savefig(real_plot_path, dpi=250)
plt.show()

print("Saved:", real_plot_path)

## 12. Summary table

This gives simple diagnostics for each FTI value.

In [ ]:
rows = []

for val, wf in scan.items():
    hp = wf["hp"]

    if np.isclose(val, 0.0):
        max_abs_dphase = 0.0
        rms_dphase = 0.0
        max_abs_relamp = 0.0
        rms_relamp = 0.0
    else:
        dphase = np.unwrap(np.angle(hp[phase_mask] / hp_gr[phase_mask]))
        rel_amp = (np.abs(hp[relamp_mask]) - np.abs(hp_gr[relamp_mask])) / np.abs(hp_gr[relamp_mask])

        max_abs_dphase = float(np.nanmax(np.abs(dphase)))
        rms_dphase = float(np.sqrt(np.nanmean(dphase**2)))
        max_abs_relamp = float(np.nanmax(np.abs(rel_amp)))
        rms_relamp = float(np.sqrt(np.nanmean(rel_amp**2)))

    rows.append({
        "fti_parameter_input": fti_parameter,
        "fti_parameter_canonical": canonical_fti_name(fti_parameter),
        "fti_value": float(val),
        "max_abs_phase_diff_rad": max_abs_dphase,
        "rms_phase_diff_rad": rms_dphase,
        "max_abs_relative_amp_diff": max_abs_relamp,
        "rms_relative_amp_diff": rms_relamp,
    })

summary = pd.DataFrame(rows)
summary_path = outdir / f"summary_{canonical_fti_name(fti_parameter)}.csv"
summary.to_csv(summary_path, index=False)

display(summary)
print("Saved:", summary_path)

## 13. Optional: scan all standard FTI parameters one by one

Keep this cell commented if you only need the second/`dchi2` parameter. Uncomment it if you want to generate one folder per FTI parameter.

This is useful for checking which FTI parameter gives the strongest phase distortion for your source.

In [ ]:
# all_fti_parameters = [
#     "fti_dchiMinus2v",
#     "fti_dchi0v",
#     "fti_dchi1v",
#     "fti_dchi2v",
#     "fti_dchi3v",
#     "fti_dchi4v",
#     "fti_dchi5v",
#     "fti_dchi5lv",
#     "fti_dchi6v",
#     "fti_dchi6lv",
#     "fti_dchi7v",
# ]
#
# all_summary_rows = []
#
# for par in all_fti_parameters:
#     print("\n" + "=" * 80)
#     print("Scanning", par)
#     print("=" * 80)
#
#     # Reuse the same fti_values array.
#     gr_local = generate_seobnrv5hmrom_fd(
#         source=source,
#         f_min=f_min,
#         f_max=f_max,
#         delta_f=delta_f,
#         f_ref=f_ref,
#         fti_parameter=None,
#         fti_value=0.0,
#         modes=modes,
#     )
#     hp_gr_local = gr_local["hp"]
#     f_local = gr_local["f"]
#     amp_threshold_local = 1e-8 * np.nanmax(np.abs(hp_gr_local))
#     phase_mask_local = (f_local >= f_min) & (f_local <= f_max) & (np.abs(hp_gr_local) > amp_threshold_local)
#
#     for val in fti_values:
#         wf = generate_seobnrv5hmrom_fd(
#             source=source,
#             f_min=f_min,
#             f_max=f_max,
#             delta_f=delta_f,
#             f_ref=f_ref,
#             fti_parameter=par,
#             fti_value=val,
#             modes=modes,
#         )
#
#         if np.isclose(val, 0.0):
#             max_abs_dphase = 0.0
#             rms_dphase = 0.0
#         else:
#             dphase = np.unwrap(np.angle(wf["hp"][phase_mask_local] / hp_gr_local[phase_mask_local]))
#             max_abs_dphase = float(np.nanmax(np.abs(dphase)))
#             rms_dphase = float(np.sqrt(np.nanmean(dphase**2)))
#
#         all_summary_rows.append({
#             "fti_parameter_input": par,
#             "fti_parameter_canonical": canonical_fti_name(par),
#             "fti_value": float(val),
#             "max_abs_phase_diff_rad": max_abs_dphase,
#             "rms_phase_diff_rad": rms_dphase,
#         })
#
# all_summary = pd.DataFrame(all_summary_rows)
# all_summary_path = outdir / "summary_all_fti_parameters.csv"
# all_summary.to_csv(all_summary_path, index=False)
# display(all_summary)
# print("Saved:", all_summary_path)

## 14. Notes for your project

- This notebook uses the same model for GR and FTI waveforms: `SEOBNRv5HMROM`.
- Therefore, the GR case `fti_value = 0` is exactly the baseline for this scan.
- The FTI parameter is applied as a frequency-domain phase deformation through the LALSimulation non-GR `dchi` insertion functions.
- For a pure FTI phase scan, the most important figure is usually the phase-difference plot.
- If you later want the LISA `A/E/T` response, you can feed each generated `hp/hc` FD waveform into your existing lisabeta response machinery, or replace the `generate_seobnrv5hmrom_fd` function by the same wrapper you use in `build_wftdi_from_hlm_fd`.